# 🧠 nano-KAN: GPT-2 + B-spline KAN

**nanoGPT с заменой MLP на HybridKANMLP (параллельный KAN + MLP)**

| | |
|---|---|
| Attention | Стандартный causal multi-head (без изменений) |
| MLP | → `HybridKANMLP`: `out = MLP(x) + KAN(x)` |
| KAN | B-spline на сетке (grid=5, order=3) |
| Модель | 256d, 6 layers, 4 heads, ~21M params |
| Датасет | Shakespeare (tiktoken GPT-2 BPE) |

> ⚡ **Runtime → Change runtime type → GPU (T4)** перед запуском!

## 1. Клонирование репо и проверка GPU

In [ ]:
!git clone https://github.com/icas711/nano-kan.git
%cd nano-kan

!pip install tiktoken -q

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Memory: {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU не подключён! Runtime → Change runtime type → GPU")

## 2. Подготовка данных (Shakespeare)

In [ ]:
!python data/shakespeare/prepare.py

## 3. Обучение

5000 итераций, batch=32×2 (grad accum), cosine LR, mixed precision. На T4 GPU ~10-15 мин.

In [ ]:
!python train.py --device cuda --max_iters 5000 --batch_size 32 --grad_accum_steps 2

## 4. Генерация текста

In [ ]:
!python generate.py --prompt "To be, or not to be," --max_tokens 500 --temperature 0.8

## 5. Интерактивная генерация

Попробуйте свои промпты:

In [ ]:
import sys
sys.path.insert(0, ".")
import torch
import tiktoken
from model import GPT

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load("out/ckpt.pt", map_location=device, weights_only=False)
model = GPT(ckpt["config"]).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
enc = tiktoken.get_encoding("gpt2")

def generate(prompt: str, max_tokens=300, temperature=0.8, top_k=200):
    tokens = enc.encode(prompt, allowed_special=set())
    x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        y = model.generate(x, max_new_tokens=max_tokens, temperature=temperature, top_k=top_k)
    print(enc.decode(y[0].tolist()))

generate("O Romeo, Romeo, wherefore art thou Romeo?")

In [ ]:
generate("KING HENRY:")

# 🧠 nano-KAN: GPT-2 + B-spline KAN

**nanoGPT с заменой MLP на HybridKANMLP (параллельный KAN + MLP)**

| | |
|---|---|
| Attention | Стандартный causal multi-head (без изменений) |
| MLP | → `HybridKANMLP`: `out = MLP(x) + KAN(x)` |
| KAN | B-spline на сетке (grid=5, order=3) |
| Модель | 256d, 6 layers, 4 heads, ~21M params |
| Датасет | Shakespeare (tiktoken GPT-2 BPE) |

> ⚡ **Runtime → Change runtime type → GPU (T4)** перед запуском!

## 1. Клонирование репо и проверка GPU

In [ ]:
!git clone https://github.com/icas711/nano-kan.git
%cd nano-kan

!pip install tiktoken -q

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Memory: {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU не подключён! Runtime → Change runtime type → GPU")

## 2. Подготовка данных (Shakespeare)

In [ ]:
!python data/shakespeare/prepare.py

## 3. Обучение

5000 итераций, batch=64, cosine LR, mixed precision. На T4 GPU ~10-15 мин.

In [ ]:
!python train.py --device cuda --max_iters 5000 --batch_size 32 --grad_accum_steps 2

## 4. Генерация текста

In [ ]:
!python generate.py --prompt "To be, or not to be," --max_tokens 500 --temperature 0.8

## 5. Интерактивная генерация

Попробуйте свои промпты:

In [ ]:
import sys
sys.path.insert(0, ".")
import torch
import tiktoken
from model import GPT

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load("out/ckpt.pt", map_location=device, weights_only=False)
model = GPT(ckpt["config"]).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
enc = tiktoken.get_encoding("gpt2")

def generate(prompt: str, max_tokens=300, temperature=0.8, top_k=200):
    tokens = enc.encode(prompt, allowed_special=set())
    x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        y = model.generate(x, max_new_tokens=max_tokens, temperature=temperature, top_k=top_k)
    print(enc.decode(y[0].tolist()))

# Попробуйте:
generate("O Romeo, Romeo, wherefore art thou Romeo?")

In [ ]:
generate("KING HENRY:")

# 🧠 nano-KAN: GPT-2 + B-spline KAN

**nanoGPT с заменой MLP на HybridKANMLP (параллельный KAN + MLP)**

Архитектура:
- Attention — стандартный causal multi-head (без изменений)
- MLP → `HybridKANMLP`: `out = MLP(x) + KAN(x)`
- KAN: B-spline на сетке (grid=5, order=3)
- Модель: 256d, 6 layers, 4 heads, ~21M params
- Датасет: Shakespeare (tiktoken GPT-2 BPE)

> **Runtime → Change runtime type → GPU (T4)** перед запуском!

In [ ]:
# Проверка GPU и установка зависимостей
!nvidia-smi
!pip install tiktoken -q

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    device = "cuda"
else:
    print("⚠️ GPU не подключён! Runtime → Change runtime type → GPU")
    device = "cpu"

## 1. Скачивание и токенизация Shakespeare

In [ ]:
import os
import urllib.request
import numpy as np
import tiktoken

DATA_DIR = "data/shakespeare"
os.makedirs(DATA_DIR, exist_ok=True)

input_path = os.path.join(DATA_DIR, "input.txt")
if not os.path.exists(input_path):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print(f"Downloading Shakespeare...")
    urllib.request.urlretrieve(url, input_path)

with open(input_path, "r") as f:
    text = f.read()
print(f"Dataset: {len(text):,} chars")

enc = tiktoken.get_encoding("gpt2")
tokens = np.array(enc.encode_ordinary(text), dtype=np.uint16)
print(f"Tokens: {len(tokens):,}")

split = int(len(tokens) * 0.9)
tokens[:split].tofile(os.path.join(DATA_DIR, "train.bin"))
tokens[split:].tofile(os.path.join(DATA_DIR, "val.bin"))
print(f"Train: {split:,} | Val: {len(tokens)-split:,}")

## 2. B-spline KAN Layer

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class KANLinear(nn.Module):
    """
    B-spline KAN layer: each (input, output) edge carries a learnable
    univariate B-spline function plus a residual SiLU branch.
    out_j = sum_i [ w_base_{ij} * silu(x_i) + spline_{ij}(x_i) ]
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        grid_size: int = 5,
        spline_order: int = 3,
        scale_noise: float = 0.1,
        scale_base: float = 1.0,
        scale_spline: float = 1.0,
        grid_range: tuple = (-1.0, 1.0),
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = torch.arange(-spline_order, grid_size + spline_order + 1, dtype=torch.float32) * h + grid_range[0]
        grid = grid.unsqueeze(0).expand(in_features, -1)
        self.register_buffer("grid", grid)

        self.spline_weight = nn.Parameter(torch.empty(out_features, in_features, grid_size + spline_order))
        self.base_weight = nn.Parameter(torch.empty(out_features, in_features))

        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.scale_noise = scale_noise
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5))
        with torch.no_grad():
            noise = (torch.rand_like(self.spline_weight) - 0.5) * self.scale_noise
            x_init = torch.linspace(-1, 1, self.grid_size + self.spline_order, device=self.spline_weight.device)
            x_init = x_init.unsqueeze(0).expand(self.in_features, -1)
            self.spline_weight.copy_(x_init.unsqueeze(0).expand(self.out_features, -1, -1) * 0.1 + noise)

    def b_splines(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(-1)
        grid = self.grid
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).float()
        for k in range(1, self.spline_order + 1):
            left_num = x - grid[:, :-(k + 1)]
            left_den = grid[:, k:-1] - grid[:, :-(k + 1)]
            right_num = grid[:, k + 1:] - x
            right_den = grid[:, k + 1:] - grid[:, 1:(-k) if (-k) != 0 else None]
            left = left_num / left_den.clamp(min=1e-7)
            right = right_num / right_den.clamp(min=1e-7)
            bases = left * bases[:, :, :-1] + right * bases[:, :, 1:]
        return bases

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        original_shape = x.shape
        x_flat = x.reshape(-1, self.in_features)
        base_out = F.linear(F.silu(x_flat), self.base_weight) * self.scale_base
        splines = self.b_splines(x_flat)
        spline_out = torch.einsum("bic,oic->bo", splines, self.spline_weight) * self.scale_spline
        out = base_out + spline_out
        return out.reshape(*original_shape[:-1], self.out_features)


print("✓ KANLinear defined")

## 3. GPT + HybridKANMLP Model

In [ ]:
from dataclasses import dataclass


@dataclass
class GPTConfig:
    block_size: int = 256
    vocab_size: int = 50257
    n_layer: int = 6
    n_head: int = 4
    n_embd: int = 256
    dropout: float = 0.1
    bias: bool = False
    kan_grid_size: int = 5
    kan_spline_order: int = 3


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(F.gelu(self.c_fc(x))))


class HybridKANMLP(nn.Module):
    """Parallel: out = MLP(x) + KAN(x)"""
    def __init__(self, config):
        super().__init__()
        self.mlp = MLP(config)
        self.kan = KANLinear(config.n_embd, config.n_embd,
                             grid_size=config.kan_grid_size,
                             spline_order=config.kan_spline_order)

    def forward(self, x):
        return self.mlp(x) + self.kan(x)


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = HybridKANMLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=nn.LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

        n_params = sum(p.numel() for p in self.parameters()) - self.transformer.wpe.weight.numel()
        print(f"nano-KAN: {n_params/1e6:.2f}M parameters")

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


# Quick shape test
config = GPTConfig()
model = GPT(config).to(device)
x_test = torch.randint(0, 50257, (2, 64), device=device)
logits, loss = model(x_test, x_test)
print(f"✓ Forward pass: logits {logits.shape}, loss {loss.item():.2f}")

## 4. Обучение

Конфиг: 5000 итераций, batch=64, cosine LR schedule, mixed precision (float16).
На T4 GPU обучение занимает ~10-15 мин.

In [ ]:
import time
from contextlib import nullcontext

# ---- Hyperparameters ----
data_dir      = "data/shakespeare"
batch_size    = 64
block_size    = 256
max_iters     = 5000
eval_interval = 250
eval_iters    = 200
log_interval  = 50
lr            = 1e-3
min_lr        = 1e-4
warmup_iters  = 100
lr_decay_iters = 5000
weight_decay  = 0.1
beta1, beta2  = 0.9, 0.95
grad_clip     = 1.0
use_compile   = False   # True for extra speed (takes time to compile first)

# ---- AMP context ----
dtype = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float16"
ptdtype = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[dtype]
ctx = nullcontext() if device == "cpu" else torch.amp.autocast(device_type="cuda", dtype=ptdtype)

# ---- Data loading ----
def get_batch(split):
    fname = "train.bin" if split == "train" else "val.bin"
    data = np.memmap(os.path.join(data_dir, fname), dtype=np.uint16, mode="r")
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

def get_lr(it):
    if it < warmup_iters:
        return lr * it / warmup_iters
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (lr - min_lr)

# ---- Reinit model on GPU ----
torch.manual_seed(1337)
config = GPTConfig()
model = GPT(config).to(device)

if use_compile:
    print("Compiling model...")
    model = torch.compile(model)

# ---- Optimizer ----
param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
optimizer = torch.optim.AdamW([
    {"params": decay_params, "weight_decay": weight_decay},
    {"params": nodecay_params, "weight_decay": 0.0},
], lr=lr, betas=(beta1, beta2), fused=(device == "cuda"))
scaler = torch.amp.GradScaler("cuda", enabled=(dtype == "float16"))

# ---- Training loop ----
best_val_loss = float("inf")
train_losses = []
val_losses = []
t0 = time.time()

for it in range(max_iters):
    cur_lr = get_lr(it)
    for pg in optimizer.param_groups:
        pg["lr"] = cur_lr

    # Eval
    if it % eval_interval == 0 or it == max_iters - 1:
        losses = estimate_loss()
        train_losses.append((it, losses["train"]))
        val_losses.append((it, losses["val"]))
        print(f"step {it:5d} | train {losses['train']:.4f} | val {losses['val']:.4f}")
        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            torch.save({"model": model.state_dict(), "config": config,
                        "iter": it, "best_val_loss": best_val_loss}, "ckpt.pt")
            print(f"  → checkpoint saved (val={best_val_loss:.4f})")

    # Forward / backward
    optimizer.zero_grad(set_to_none=True)
    X, Y = get_batch("train")
    with ctx:
        _, loss = model(X, Y)
    scaler.scale(loss).backward()
    if grad_clip > 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()

    if it % log_interval == 0:
        dt = time.time() - t0
        t0 = time.time()
        print(f"  iter {it:5d} | loss {loss.item():.4f} | lr {cur_lr:.6f} | {dt*1000:.0f}ms")

print(f"\nTraining done! Best val loss: {best_val_loss:.4f}")
if torch.cuda.is_available():
    print(f"GPU memory used: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

## 5. Loss Curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(*zip(*train_losses), label="train", marker="o", markersize=4)
ax.plot(*zip(*val_losses), label="val", marker="s", markersize=4)
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.set_title("nano-KAN Training")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Генерация текста

In [ ]:
# Загрузить лучший чекпоинт
ckpt = torch.load("ckpt.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()

enc = tiktoken.get_encoding("gpt2")
prompt = "To be, or not to be,"
tokens = enc.encode(prompt, allowed_special=set())
x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)

with torch.no_grad():
    y = model.generate(x, max_new_tokens=500, temperature=0.8, top_k=200)

print("=" * 60)
print(enc.decode(y[0].tolist()))
print("=" * 60)